# AngetSDK

## Overview

#### init

In [9]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions

from dotenv import load_dotenv
load_dotenv()


True

In [3]:
async def main():
    async for message in query(
        prompt = "What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"])
    ):
        if hasattr(message, "result"):
            print(message.result)

#### built-in tools overview

In [6]:
import threading

def run_async(coro_factory):
    """ Proactor ループを別スレッドで建て、 coroutine を実行する

    Args:
        coro_factory (_type_): _description_
    """

    box = {}
    def runner():
        try:
            with asyncio.Runner(loop_factory=asyncio.ProactorEventLoop) as r:
                box["value"] = r.run(coro_factory())
        except BaseException as e:
            box["error"] = e

    t = threading.Thread(target=runner)
    t.start()
    t.join()

    if "error" in box:
        raise box["error"]

    return box.get("value")

run_async(main)


Contents of `402---CCA-F`:

**Directories**
- `.claude/` — project Claude Code config (commands, settings)
- `.git/`
- `Handson/`
- `agentSDK/`
- `reference/` — the course material (origin.md / summary.md pairs)

**Files**
- `.gitignore` (140 B)
- `CLAUDE.md` (1.3 KB) — project instructions
- `README.md` (14.7 KB)
- `SUMMARY.MD` (1.6 KB)


#### hooks overview

In [21]:
from claude_agent_sdk import HookMatcher
from datetime import datetime

from pathlib import Path

target = Path("hook_test").resolve()
target.mkdir(exist_ok=True)
prompt = f"{target / 'test.md'} に '# Claude here v3' って書いて"


async def log_file_change(input_data, tool_use_id, context):
    file_path = input_data.get("tool_input", {}).get("file_path", "unknown")
    with open("./hook_test/test.log", "a") as f:
        f.write(f"{datetime.now()} : modified {file_path} \n")

    return {}

async def hook_demo():
    async for message in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            permission_mode="acceptEdits",
            settings=str(Path("sdk_settings.json").resolve()),
            allowed_tools=["Write", "Edit"],
            disallowed_tools=["Task", "Bash", "WebSearch", "WebFetch", "Skill", "Workflow"],
            max_turns=3,
            hooks={
                "PostToolUse": [
                    HookMatcher(matcher="Edit|Write",
                                hooks=[log_file_change]
                                )
                ]
            }
        )
    ):
        # debug
        print(type(message).__name__, message)

        if hasattr(message, "result"):
            print("---message---")
            print(message.result)
            print("-------------")


In [13]:
import asyncio
import threading

def run_async(coro_factory):
    """ Proactor ループを別スレッドで建て、 coroutine を実行する

    Args:
        coro_factory (_type_): _description_
    """

    box = {}
    def runner():
        try:
            with asyncio.Runner(loop_factory=asyncio.ProactorEventLoop) as r:
                box["value"] = r.run(coro_factory())
        except BaseException as e:
            box["error"] = e

    t = threading.Thread(target=runner)
    t.start()
    t.join()

    if "error" in box:
        raise box["error"]

    return box.get("value")

In [18]:
# ClaudeAgentOptionsですべてがデフォルトの状態。(コストがめちゃくちゃ高い)
run_async(hook_demo)

SystemMessage SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': 'c:\\Users\\hakuu\\Documents\\101-programing\\402---CCA-F\\agentSDK', 'session_id': 'd4cc336e-e0d8-47f6-b563-f6d8c810c4a8', 'tools': ['Task', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'DesignSync', 'Edit', 'EnterWorktree', 'ExitWorktree', 'Glob', 'Grep', 'LSP', 'Monitor', 'NotebookEdit', 'PowerShell', 'PushNotification', 'Read', 'ReportFindings', 'ScheduleWakeup', 'SendMessage', 'Skill', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskOutput', 'TaskStop', 'TaskUpdate', 'ToolSearch', 'WebFetch', 'WebSearch', 'Workflow', 'Write'], 'mcp_servers': [], 'model': 'claude-opus-5', 'permissionMode': 'acceptEdits', 'slash_commands': ['fable-cpp-discipline', 'fable-markdown-discipline', 'fable-nextjs-discipline', 'fable-python-discipline', 'fable-rust-discipline', 'fable-skill-architect', 'fable-unity-csharp-discipline', 'commit-refined-summary', 'fable', 'refine-summary', 'deep-research', 'design-sync',

In [22]:
# after revising ClaudeAgentOptions

run_async(hook_demo)

SystemMessage SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': 'c:\\Users\\hakuu\\Documents\\101-programing\\402---CCA-F\\agentSDK', 'session_id': '28152802-1cee-4902-8e42-7436aa1dca4d', 'tools': ['CronCreate', 'CronDelete', 'CronList', 'DesignSync', 'Edit', 'EnterWorktree', 'ExitWorktree', 'Glob', 'Grep', 'LSP', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'ReportFindings', 'ScheduleWakeup', 'SendMessage', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskOutput', 'TaskStop', 'TaskUpdate', 'ToolSearch', 'Write'], 'mcp_servers': [], 'model': 'claude-haiku-4-5-20251001', 'permissionMode': 'acceptEdits', 'slash_commands': ['fable-cpp-discipline', 'fable-markdown-discipline', 'fable-nextjs-discipline', 'fable-python-discipline', 'fable-rust-discipline', 'fable-skill-architect', 'fable-unity-csharp-discipline', 'commit-refined-summary', 'fable', 'refine-summary', 'deep-research', 'design-sync', 'dataviz', 'update-config', 'verify', 'debug', 'code-review', '

#### agent overview

In [ ]:
from claude_agent_sdk import AgentDefinition

async def agent_test():
    async for message in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob", "Grep", "Agent"],
            agents={
                "code-reviewer" : AgentDefinition(
                    description="Expert code reviewer for qulaity and security reviews.",
                    prompt="",
                    tools=["Read", "Glob", "Grep"]
                )
            }
        )
    ):
        if hasattr(message, "result"):
            print(message.result)

#### session overview

In [ ]:
from claude_agent_sdk import SystemMessage, ResultMessage

async def session_test():

    session_id = None

    async for message in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Grep"]
        )
    ):
        if isinstance(message, SystemMessage) and message.subtype == "init":
            session_id = message.data["session_id"]

    async for message in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            resume=session_id
        )
    ):
        if isinstance(message, ResultMessage):
            print(message.result)


### Hook 
https://code.claude.com/docs/ja/agent-sdk/hooks

In [ ]:
import asyncio
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeSDKClient,
    ClaudeAgentOptions,
    HookMatcher,
    ResultMessage,
)

async def protect_env_files(input_data, tool_use_id, context):
    file_path = input_data["tool_input"].get("input_path", "")
    file_name = file_path.split("/")[-1]

    if file_name == ".env":
        return {
            "hookSpecificOutput": {
                "hookEventName" : input_data["hook_event_name"],
                "permissionDecision": "deny",
                "permissionDecisionReason": "Cannot modify .env files"
            }
        }
    return {}

async def main():
    options = ClaudeAgentOptions(
        hooks={
            "PreToolUse": [
                HookMatcher(
                    matcher="Write|Edit",
                    hooks=[protect_env_files]
                )
            ]
        }
    )

    async with ClaudeSDKClient(options=options) as client:
        await client.query("Update the database configuration")
        async for message in client.receive_responose():
            if isinstance(message, (AssistantMessage, ResultMessage)):
                print(message)
